# Part 3 — Fine-tune PhoBERT cho Aspect Classification (UIT-ViSFD)

**Mục tiêu:** Thay thế BiLSTM (embedding tĩnh) bằng PhoBERT fine-tuned với Classification Head.
Kết quả kỳ vọng: micro-F1 tăng từ ~0.79 → ≥0.85.

**Chạy trên Kaggle:** GPU T4 (miễn phí, 16 GB VRAM). Thời gian ước tính ~35 phút (5 epoch).

## Tại sao PhoBERT tốt hơn BiLSTM?
| | BiLSTM | PhoBERT fine-tuned |
|---|---|---|
| Ngữ cảnh | Local (cửa sổ LSTM) | Global (Transformer attention) |
| Biểu diễn | Tĩnh (embedding cố định) | Động (context-aware) |
| Pre-training | Không | 20 GB văn bản tiếng Việt |
| Tham số | ~2.5M | ~135M (frozen backbone) / ~10M (head only) |

---

In [ ]:
# Cài dependencies (Kaggle đã có transformers/torch)
!pip install underthesea -q
!pip install rank-bm25 -q

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import f1_score, precision_score, recall_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## 1. Tải dữ liệu UIT-ViSFD

In [ ]:
ASPECTS = [
    'SCREEN', 'CAMERA', 'BATTERY', 'PERFORMANCE',
    'STORAGE', 'DESIGN', 'PRICE', 'GENERAL', 'FEATURES', 'SER&ACC'
]
ASPECT_PATTERN = re.compile(r'\{([\w&]+)#(\w+)\}')

def parse_label(label_str):
    if pd.isna(label_str):
        return []
    return [(m.group(1), m.group(2)) for m in ASPECT_PATTERN.finditer(str(label_str))]

def load_split(filename):
    # Thử Kaggle input path trước, fallback về relative
    for base in [Path('/kaggle/input/uit-visfd'), Path('data/raw'), Path('../data/raw')]:
        p = base / filename
        if p.exists():
            break
    df = pd.read_csv(p)
    df['parsed'] = df['label'].apply(parse_label)
    df['aspects'] = df['parsed'].apply(lambda x: [a for a, _ in x])
    return df

train_df = load_split('Train.csv')
dev_df   = load_split('Dev.csv')
test_df  = load_split('Test.csv')

print(f'Train: {len(train_df)}  Dev: {len(dev_df)}  Test: {len(test_df)}')
print(f'Aspects: {ASPECTS}')

## 2. Dataset class

In [ ]:
MODEL_NAME = 'vinai/phobert-base-v2'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN    = 128

class ViSFDDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts    = df['comment'].astype(str).tolist()
        self.aspects  = df['aspects'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        label = torch.zeros(len(ASPECTS))
        for a in self.aspects[idx]:
            if a in ASPECTS:
                label[ASPECTS.index(a)] = 1.0
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         label,
        }

train_ds = ViSFDDataset(train_df, tokenizer, MAX_LEN)
dev_ds   = ViSFDDataset(dev_df,   tokenizer, MAX_LEN)
test_ds  = ViSFDDataset(test_df,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
dev_loader   = DataLoader(dev_ds,   batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2)
print('DataLoaders OK')

## 3. Mô hình PhoBERT + Classification Head

Kiến trúc:
```
PhoBERT-base-v2 (768d)
  └─ [CLS] token → Dropout(0.3) → Linear(768 → 256) → ReLU → Linear(256 → 10) → Sigmoid
```
Backbone PhoBERT được **fine-tune toàn bộ** (không frozen) với learning rate nhỏ.

In [ ]:
class PhoBERTAspectClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, n_aspects=10, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.bert.config.hidden_size  # 768 for phobert-base
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, n_aspects),
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # [CLS] token
        return self.head(cls)

model = PhoBERTAspectClassifier().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {n_params/1e6:.1f}M  Trainable: {n_train/1e6:.1f}M')

## 4. Training với Linear Warmup + Cosine Decay

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

EPOCHS    = 5
LR        = 2e-5   # phù hợp cho BERT fine-tuning
THRESHOLD = 0.5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = OneCycleLR(
    optimizer,
    max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1,   # 10% warmup
)
loss_fn = nn.BCEWithLogitsLoss()

def evaluate(loader):
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model(ids, mask)
            pred = (torch.sigmoid(logits) >= THRESHOLD).cpu().numpy()
            true = batch['labels'].numpy().astype(int)
            all_pred.append(pred)
            all_true.append(true)
    P = np.vstack(all_pred)
    T = np.vstack(all_true)
    micro_f1 = f1_score(T, P, average='micro', zero_division=0)
    macro_f1 = f1_score(T, P, average='macro', zero_division=0)
    per_f1   = f1_score(T, P, average=None,    zero_division=0)
    return micro_f1, macro_f1, per_f1

history = []
best_micro = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = loss_fn(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    micro, macro, per = evaluate(dev_loader)
    history.append({'epoch': epoch, 'loss': avg_loss, 'micro_f1': micro, 'macro_f1': macro})
    print(f'Epoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  micro-F1={micro:.4f}  macro-F1={macro:.4f}')

    if micro > best_micro:
        best_micro = micro
        torch.save(model.state_dict(), '/kaggle/working/phobert_aspect_best.pt')
        print(f'  -> Saved best (micro-F1={best_micro:.4f})')

## 5. Đánh giá trên Test set

In [ ]:
# Nạp best checkpoint
model.load_state_dict(torch.load('/kaggle/working/phobert_aspect_best.pt', map_location=DEVICE))

test_micro, test_macro, test_per = evaluate(test_loader)
dev_micro,  dev_macro,  dev_per  = evaluate(dev_loader)

print(f'\n=== KẾT QUẢ CUỐI (best checkpoint) ===')
print(f'  Dev  — micro-F1: {dev_micro:.4f}   macro-F1: {dev_macro:.4f}')
print(f'  Test — micro-F1: {test_micro:.4f}   macro-F1: {test_macro:.4f}')
print(f'\nPer-aspect F1 trên Test:')
print(f'{"Aspect":14} {"F1":>7}')
print('-' * 24)
for asp, f1 in zip(ASPECTS, test_per):
    print(f'{asp:14} {f1:.4f}')

## 6. So sánh với BiLSTM baseline

In [ ]:
import matplotlib.pyplot as plt

# BiLSTM baseline (đã có trong artifacts/metrics.json)
bilstm_micro = 0.7907  # cập nhật từ artifacts/metrics.json

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Learning curve
epochs = [h['epoch'] for h in history]
axes[0].plot(epochs, [h['micro_f1'] for h in history], 'b-o', label='micro-F1')
axes[0].plot(epochs, [h['macro_f1'] for h in history], 'r--s', label='macro-F1')
axes[0].axhline(bilstm_micro, color='gray', linestyle=':', label=f'BiLSTM baseline ({bilstm_micro:.4f})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('F1')
axes[0].set_title('PhoBERT Fine-tuning — Learning Curve (Dev)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Biểu đồ 2: Per-aspect F1 so sánh
x = np.arange(len(ASPECTS))
axes[1].bar(x - 0.2, test_per, 0.4, label='PhoBERT fine-tuned', color='steelblue')
axes[1].set_xticks(x)
axes[1].set_xticklabels(ASPECTS, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Per-aspect F1 — PhoBERT fine-tuned (Test set)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('/kaggle/working/phobert_finetuned_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: phobert_finetuned_results.png')

## 7. Lưu checkpoint theo định dạng deploy của project

Để deploy vào `vngraphrag`, cần lưu checkpoint theo đúng format `AspectClassifier.load()` đọc.
**Lưu ý:** Cần update `aspect_clf.py` để hỗ trợ loại model mới (xem hướng dẫn bên dưới).

In [ ]:
# Lưu đầy đủ cho deploy
deploy_ckpt = {
    'model_type':  'phobert_finetuned',
    'model_name':  MODEL_NAME,
    'state_dict':  {k: v.cpu() for k, v in model.state_dict().items()},
    'aspects':     ASPECTS,
    'max_len':     MAX_LEN,
    'threshold':   THRESHOLD,
    'dev_micro_f1':  dev_micro,
    'test_micro_f1': test_micro,
    'test_macro_f1': test_macro,
    'test_per_aspect_f1': dict(zip(ASPECTS, test_per.tolist())),
}

torch.save(deploy_ckpt, '/kaggle/working/aspect_clf_phobert.pt')

# Lưu metrics để đưa vào báo cáo
metrics = {
    'model':       'PhoBERT-base-v2 fine-tuned',
    'dataset':     'UIT-ViSFD',
    'dev': {'micro_f1': round(dev_micro, 4), 'macro_f1': round(dev_macro, 4)},
    'test':{
        'micro_f1': round(test_micro, 4),
        'macro_f1': round(test_macro, 4),
        'per_aspect_f1': {a: round(float(f), 4) for a, f in zip(ASPECTS, test_per)},
    },
    'vs_bilstm': {
        'bilstm_micro_f1':   bilstm_micro,
        'improvement_abs':   round(test_micro - bilstm_micro, 4),
        'improvement_rel_%': round((test_micro - bilstm_micro) / bilstm_micro * 100, 1),
    },
    'history': history,
}
with open('/kaggle/working/phobert_finetuned_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('Saved:')
print('  /kaggle/working/aspect_clf_phobert.pt')
print('  /kaggle/working/phobert_finetuned_metrics.json')
print('  /kaggle/working/phobert_finetuned_results.png')
print(f'\nImprovement vs BiLSTM: +{metrics["vs_bilstm"]["improvement_abs"]:.4f} '
      f'({metrics["vs_bilstm"]["improvement_rel_%"]:+.1f}%)')

## 8. Hướng dẫn tích hợp vào project

### Bước 1 — Download artifacts từ Kaggle
```bash
# Sau khi notebook chạy xong, download từ Output:
kaggle kernels output <your-username>/kaggle-part3 -p artifacts/
# Hoặc download thủ công từ giao diện Kaggle
```

### Bước 2 — Cập nhật `aspect_clf.py` để hỗ trợ PhoBERT checkpoint

Mở [src/vngraphrag/core/aspect_clf.py](../src/vngraphrag/core/aspect_clf.py) và thêm loader cho `model_type == 'phobert_finetuned'`:

```python
@classmethod
def load(cls, artifacts_dir, threshold=0.5):
    p = Path(artifacts_dir) / ASPECT_CLF_FILE
    if not p.exists():
        return None
    import torch
    ckpt = torch.load(p, map_location='cpu')
    
    if ckpt.get('model_type') == 'phobert_finetuned':
        # PhoBERT fine-tuned path
        from transformers import AutoModel, AutoTokenizer
        bert = AutoModel.from_pretrained(ckpt['model_name'])
        # ... rebuild PhoBERTAspectClassifier, load state_dict
    else:
        # BiLSTM path (backward compat)
        model = _build_model(...)
```

### Bước 3 — Đặt file checkpoint
```bash
cp /kaggle/working/aspect_clf_phobert.pt artifacts/aspect_clf.pt
make eval  # kiểm tra CI gate
```

---

## Kết luận

PhoBERT fine-tuned đạt micro-F1 cao hơn BiLSTM nhờ:
1. **Contextualized embedding**: cùng từ "tốt" trong "camera tốt" vs "dịch vụ tốt" → vector khác nhau
2. **Self-attention**: bắt được quan hệ xa trong câu
3. **Pre-trained knowledge**: 20 GB văn bản tiếng Việt → hiểu idiom/slang e-commerce

Kết quả này trực tiếp cải thiện chất lượng Knowledge Graph (aspect prediction cho Shopee reviews)
và endpoint `POST /classify`.